## Week 5 Day 5: Solar System RAG Chatbot

**Overview:**
This exercise implements an end-to-end Retrieval-Augmented Generation (RAG) system that answers questions about the Solar System using data directly sourced from Wikipedia.

**Workflow:**
1. **Data Ingestion**: Get Solar System Wikipedia page data using wikipediaapi and  segment it into semantic sections, and chunk the text.
2. **Vector Database**: Generate text embeddings and store the chunks in a local Chroma vector database.
3. **Data Visualization**: Use t-SNE to reduce high-dimensional embeddings and visualize the knowledge base in interactive 2D and 3D Plotly graphs.
4. **Advanced RAG Setup**:
   - *Query Rewriting*: Refine user questions for optimal retrieval.
   - *Retrieval & Reranking*: Fetch relevant chunks and rerank them using an LLM.
   - *Generation*: Produce accurate and context-aware answers to user queries.
5. **Interactive Chat UI**: Launch a Gradio chat application to converse with the Solar System knowledge base.

In [ ]:
# imports
import gradio as gr
import numpy as np
import plotly.graph_objects as go
from ingest import run_ingestion, DB_NAME, collection_name
from answer import answer_question
from chromadb import PersistentClient
from sklearn.manifold import TSNE

In [ ]:
# Run ingestion
run_ingestion()

## Visualize Vector Store

In [ ]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["type"] for metadata in metadatas]

colors = [
    [
        "blue", "green", "red", "orange", "purple",
        "cyan", "magenta", "yellow", "black", "gray",
        "pink", "brown", "olive", "teal", "navy"
    ][
        [
            "boundary_region_and_uncertainties", "celestial_neighborhood",
            "definition", "discovery_and_exploration", "external_links",
            "formation_and_evolution", "galactic_position", "general_characteristics",
            "gravitationally_unstable_populations", "inner_solar_system",
            "outer_solar_system", "references", "see_also", "sun",
            "trans-neptunian_region"
        ].index(t)
    ]
    for t in doc_types
]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()

In [ ]:
# Gradio Interface
def chat_fn(message, history):
    formatted_history = []
    for user_msg, bot_msg in history:
        formatted_history.append({"role": "user", "content": user_msg})
        formatted_history.append({"role": "assistant", "content": bot_msg})
    
    # answer_question returns (answer, context_chunks)
    answer, _ = answer_question(message, formatted_history)
    return answer

demo = gr.ChatInterface(
    fn=chat_fn,
    title="SolarFacts Knowledge Base Chat",
    description="Ask me any question about the Solar System and I will answer based on the knowledge base."
)

if __name__ == "__main__":
    demo.launch(inbrowser=True, share=False)